# AWS Bedrock AgentCore Session Management

This notebook deploys a MCP server, and then run various tests to demostrate session management best practice to achieve the best throughput. 

### MCP Server
The MCP Server has two simple tools as shown here. Each tool perform addition, sleep 2 seconds, returns the result, together with SERVER_INSTANCE_ID. The SERVER_INSTANCE_ID is a UUID, which will be used in test to determine if the code runs on the same server instance
```
SERVER_INSTANCE_ID = str(uuid.uuid4())
mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def add_numbers_sync(a: int, b: int) -> str:
    """Add two numbers together (sync)"""
    print(f"Adding numbers (sync): {a} + {b}")
    result = a + b
    time.sleep(2)
    return f"result={result} server={SERVER_INSTANCE_ID}"


@mcp.tool()
async def add_numbers_async(a: int, b: int) -> str:
    """Add two numbers together (async)"""
    print(f"Adding numbers (async): {a} + {b}")
    result = a + b
    await asyncio.sleep(2)
    return f"result={result} server={SERVER_INSTANCE_ID}"

```

### Tests
- Invoke a sync MCP tool in parallel, each tool call uses an unique session
- Invoke a sync MCP tool in parallel, all tool calls share the same session
- Invoke an async MCP tool in parallel, each tool call uses an unique session
- Invoke an async MCP tool in parallel, all tool calls share the same session


Based on the testing results, we analysis AgentCore session behaivor, and recommend best approach to build scalable system on AgentCore Runtime

## Install Dependencies

In [20]:
#!pip install --force-reinstall -U -r requirements.txt --quiet

## Setup

In [5]:
from bedrock_agentcore_starter_toolkit import Runtime
from bedrock_agentcore_starter_toolkit.operations.runtime import destroy_bedrock_agentcore
from boto3.session import Session
from pathlib import Path
import os

boto_session = Session()
region = boto_session.region_name

agentcore_control_client = boto_session.client("bedrock-agentcore-control", region_name=region)
ssm_client = boto_session.client("ssm", region_name=region)

tool_name = "blog_mcp_math"

print(f"Using AWS region: {region}")

Using AWS region: us-west-2


## Setup AgentCore MCP Server

In [6]:
# configure and launch the MCP server agent
required_files = ["mcp_server.py"]
agentcore_runtime = Runtime()
response = agentcore_runtime.configure(
    entrypoint="mcp_server.py", 
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=region,
    protocol="MCP",
    agent_name=tool_name,
)

launch_result = agentcore_runtime.launch()
print(f"Launch completed. Agent ARN: {launch_result.agent_arn}")

# store the agent ARN in Parameter Store for later retrieval by the test script
ssm_client.put_parameter(
    Name="/blog_mcp_math/runtime_iam/agent_arn",
    Value=launch_result.agent_arn,
    Type="String",
    Description="Agent ARN for blog_mcp_math MCP server",
    Overwrite=True,
)


Entrypoint parsed: file=/Users/mingyyzz/dev-local/aws.blog1/src/mcp_server.py, bedrock_agentcore_name=mcp_server
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: blog_mcp_math
Memory disabled
Network mode: PUBLIC


📄 Using existing Dockerfile: /Users/mingyyzz/dev-local/aws.blog1/src/Dockerfile

Generated .dockerignore: /Users/mingyyzz/dev-local/aws.blog1/src/.dockerignore
Keeping 'blog_mcp_math' as default agent
Bedrock AgentCore configured: /Users/mingyyzz/dev-local/aws.blog1/src/.bedrock_agentcore.yaml
🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'blog_mcp_math' to account 980413094772 (us-west-2)
Generated image tag: 20260322-172750-132
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: blog_mcp_math
ECR repository available: 980413094772.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-blog_mcp_math
Getting or creating execution role for agent: blog_mcp_math
Usin

✅ Reusing existing ECR repository: 980413094772.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-blog_mcp_math


✅ Reusing existing execution role: arn:aws:iam::980413094772:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-622e8ad068
Execution role available: arn:aws:iam::980413094772:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-622e8ad068
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: blog_mcp_math
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-622e8ad068
Reusing existing CodeBuild execution role: arn:aws:iam::980413094772:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-622e8ad068
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: blog_mcp_math/source.zip
Updated CodeBuild project: bedrock-agentcore-blog_mcp_math-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.1s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 7.4s
🔄 DOWNLOAD_SOURCE started (total: 8s)
✅ DOWNLOAD_SOURCE

Launch completed. Agent ARN: arn:aws:bedrock-agentcore:us-west-2:980413094772:runtime/blog_mcp_math-Z057Lr3Pdg


{'Version': 5,
 'Tier': 'Standard',
 'ResponseMetadata': {'RequestId': '28ebb7b9-231e-44b7-b7d6-832be500c7db',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'server': 'Server',
   'date': 'Sun, 22 Mar 2026 17:28:22 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '31',
   'connection': 'keep-alive',
   'x-amzn-requestid': '28ebb7b9-231e-44b7-b7d6-832be500c7db',
   'cache-control': 'no-store'},
  'RetryAttempts': 0}}

## Sync MCP tool Parallel Test

### Call the sync tool with 5 parallel threads, each call uses unique session -> Low Cold Start Latency

Below test result shows all calls start at the same time, the average duration is around 3 seconds, and each call is handled by different server. The key finding is AgentCore use 5 different instance, each instance has very low cold start latency.

In [4]:
!python mcp_test.py sync --calls 5 --session unique


add_numbers_sync — 5 concurrent calls, unique sessions
  add_numbers_sync #0   start=10:20:30, end=10:20:33, duration=  3.00s, server=2a55175b-187a-49e3-b640-625f168af94e
  add_numbers_sync #1   start=10:20:30, end=10:20:33, duration=  2.98s, server=a523cfb8-5836-4ba4-98ad-4449b0027176
  add_numbers_sync #2   start=10:20:30, end=10:20:33, duration=  3.02s, server=2d8b828f-2293-46ea-b1a3-1cfc89cc962c
  add_numbers_sync #4   start=10:20:30, end=10:20:33, duration=  3.01s, server=b5be6d96-238f-4dbb-808c-453fd9498c29
  add_numbers_sync #3   start=10:20:30, end=10:20:34, duration=  3.27s, server=13c32c71-5c7a-4747-8549-98e4feba9b06
  total wall time: 3.32s  avg per call: 3.06s
  unique server instances: 5


### Invoke the same tool with 15 parallel threads, some invocations has High Cold Start Latency

- The first 10 invocations have duration around 3 seconds. They have very low latency (as the tool itself sleeps 2 seconds)
- The last 5 invocations have duration around 11 seconds. They have high cold start latency
 

In [7]:
!python mcp_test.py sync --calls 15 --session unique


add_numbers_sync — 15 concurrent calls, unique sessions
  add_numbers_sync #4   start=10:35:10, end=10:35:13, duration=  3.05s, server=5375d49f-3e23-4fbf-8632-39013d01b579
  add_numbers_sync #8   start=10:35:10, end=10:35:13, duration=  3.03s, server=754ec58f-df86-42ae-a859-f2ef833add10
  add_numbers_sync #3   start=10:35:10, end=10:35:13, duration=  3.08s, server=cbb8682a-debd-4f4b-aabb-a88106c74594
  add_numbers_sync #0   start=10:35:10, end=10:35:13, duration=  3.17s, server=af7f0ed9-fa64-4ecc-b9df-bfcc7a1f75cc
  add_numbers_sync #9   start=10:35:10, end=10:35:13, duration=  3.03s, server=da8adb0c-e7e9-465c-8976-2214b8f83502
  add_numbers_sync #2   start=10:35:10, end=10:35:13, duration=  3.11s, server=e58b0778-64d7-4cfa-ba7a-482c5bf25402
  add_numbers_sync #13  start=10:35:10, end=10:35:13, duration=  3.01s, server=91d4ed92-355d-45f5-9e30-f265ae05de2f
  add_numbers_sync #6   start=10:35:10, end=10:35:13, duration=  3.11s, server=fb886245-eb37-43d7-85fe-eeb7c3b31965
  add_numbers_s

### Invoke the same tool with same session

It's good that all invocations are served by the same AgentCore Runtime instance, 15 calls tool 31 seconds. It's not recommended to use Shared Session for Sync Tools.


In [10]:
!python mcp_test.py sync --calls 15 --session shared 


add_numbers_sync — 15 concurrent calls, shared Mcp-Session-Id=a1f57bca-fbd5-4614-9faa-f5966b193f16
  add_numbers_sync #11  start=10:41:22, end=10:41:42, duration= 20.89s, server=1ca5deec-5203-4c57-b267-ef48a3faac4c
  add_numbers_sync #12  start=10:41:22, end=10:41:42, duration= 20.88s, server=1ca5deec-5203-4c57-b267-ef48a3faac4c
  add_numbers_sync #10  start=10:41:21, end=10:41:43, duration= 21.04s, server=1ca5deec-5203-4c57-b267-ef48a3faac4c
  add_numbers_sync #6   start=10:41:21, end=10:41:45, duration= 23.09s, server=1ca5deec-5203-4c57-b267-ef48a3faac4c
  add_numbers_sync #4   start=10:41:21, end=10:41:53, duration= 31.14s, server=1ca5deec-5203-4c57-b267-ef48a3faac4c
  add_numbers_sync #9   start=10:41:21, end=10:41:53, duration= 31.10s, server=1ca5deec-5203-4c57-b267-ef48a3faac4c
  add_numbers_sync #1   start=10:41:21, end=10:41:53, duration= 31.18s, server=1ca5deec-5203-4c57-b267-ef48a3faac4c
  add_numbers_sync #5   start=10:41:21, end=10:41:53, duration= 31.14s, server=1ca5deec-

## Async MCP Tool Parallel Tests

### Invoke Async MCP Tool with 5 parallel threads, each call use unique session -> Low Cold Start Latency

AgentCore uses 5 instances from the Warm Pool to serve the 5 parallel requests, average per call is below 3 seconds. Each invocation has very low cold start latency because the instances are from the Warm Pool.


In [11]:
!python mcp_test.py async --calls 5 --session unique


add_numbers_async — 5 concurrent calls, unique sessions
  add_numbers_async #4   start=10:46:37, end=10:46:40, duration=  2.88s, server=1e23d15d-354f-4ba8-af7e-be5d8784312a
  add_numbers_async #3   start=10:46:37, end=10:46:40, duration=  2.94s, server=2f9b327f-6137-4ed7-9788-d6a4be911e71
  add_numbers_async #2   start=10:46:37, end=10:46:40, duration=  2.98s, server=d9886c4f-8d57-48f5-9460-cea8f70ca9c3
  add_numbers_async #1   start=10:46:37, end=10:46:40, duration=  3.00s, server=46b2fe6f-808e-4991-a312-a7469c3982e3
  add_numbers_async #0   start=10:46:37, end=10:46:40, duration=  3.09s, server=12c094d8-0f09-4c4b-9a5c-9061380c88d8
  total wall time: 3.09s  avg per call: 2.98s
  unique server instances: 5


### Invoke with 15 parallel threads with unique session

- The first 10 invocations were served by the instance from Warm Pool, resulted very low cold start latency
- The last 5 invocations were served by new instances, resulted high cold start latency


In [16]:
!python mcp_test.py async --calls 30 --session unique 


add_numbers_async — 30 concurrent calls, unique sessions
  add_numbers_async #0   start=10:56:42, end=10:56:45, duration=  3.15s, server=c687bfef-a74d-4555-a114-35ec6ac6e4aa
  add_numbers_async #28  start=10:56:42, end=10:56:45, duration=  2.91s, server=f6476492-918a-4c76-a1b2-71e55578cbc0
  add_numbers_async #6   start=10:56:42, end=10:56:45, duration=  3.10s, server=c25996ac-f5be-441d-83ca-18b2f6c96414
  add_numbers_async #14  start=10:56:42, end=10:56:45, duration=  3.03s, server=55fa9d52-a881-4077-ab35-6d7b39a8f616
  add_numbers_async #10  start=10:56:42, end=10:56:45, duration=  3.07s, server=47d69194-f8dd-4e1f-a16f-4c3cec5b401f
  add_numbers_async #8   start=10:56:42, end=10:56:45, duration=  3.09s, server=a4590870-701f-4c99-a5ce-0df85033f5dd
  add_numbers_async #24  start=10:56:42, end=10:56:45, duration=  2.96s, server=54a7541f-002b-46a0-b598-cccc87838e13
  add_numbers_async #4   start=10:56:42, end=10:56:45, duration=  3.13s, server=e51cba64-e72d-4a0a-a837-e6cb848f56f0
  add_

### Invoke with 15 parallel threads using Shared Session

In [ ]:
!python mcp_test.py async --calls 15 --session shared


add_numbers_async — 15 concurrent calls, shared Mcp-Session-Id=409f874c-5e6c-47a6-b171-9ca2994a7dc8
  add_numbers_async #1   start=12:03:19, end=12:03:22, duration=  2.99s, server=0d22d2f0-2aa0-4a13-a3d2-d275cb561898
  add_numbers_async #11  start=12:03:19, end=12:03:22, duration=  2.95s, server=0d22d2f0-2aa0-4a13-a3d2-d275cb561898
  add_numbers_async #9   start=12:03:19, end=12:03:22, duration=  2.97s, server=0d22d2f0-2aa0-4a13-a3d2-d275cb561898
  add_numbers_async #6   start=12:03:19, end=12:03:22, duration=  3.00s, server=0d22d2f0-2aa0-4a13-a3d2-d275cb561898
  add_numbers_async #12  start=12:03:19, end=12:03:22, duration=  2.97s, server=0d22d2f0-2aa0-4a13-a3d2-d275cb561898
  add_numbers_async #14  start=12:03:19, end=12:03:22, duration=  2.95s, server=0d22d2f0-2aa0-4a13-a3d2-d275cb561898
  add_numbers_async #10  start=12:03:19, end=12:03:22, duration=  3.01s, server=0d22d2f0-2aa0-4a13-a3d2-d275cb561898
  add_numbers_async #0   start=12:03:19, end=12:03:22, duration=  3.14s, server=

## Cleanup (Optional)

In [ ]:
# Uncomment to clean up resources
# try:
#     ssm_client.delete_parameter(Name="/blog_mcp_math/runtime_iam/agent_arn")
#     print("Parameter Store parameter deleted")
# except ssm_client.exceptions.ParameterNotFound:
#     print("Parameter Store parameter not found")

# destroy_bedrock_agentcore(
#     config_path=Path(".bedrock_agentcore.yaml"),
#     agent_name=tool_name,
#     delete_ecr_repo=True,
# )